# Phase 1 — Data Cleaning

## Purpose

This notebook downloads the Roman Urdu sentiment dataset, checks its quality,
and creates a cleaned CSV file. The cleaning is deliberately conservative:
Roman Urdu has many valid spellings, so we do **not** try to correct spelling or
remove common words.

In [1]:
from pathlib import Path
import os

# This makes the notebook work whether it is opened from the project folder
# or from inside the notebooks folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
IMAGES = PROJECT_ROOT / "images"

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
IMAGES.mkdir(parents=True, exist_ok=True)

# Store the downloaded Hugging Face files inside this project.
os.environ["HF_HOME"] = str(PROJECT_ROOT / ".cache" / "huggingface")

print("Project folder:", PROJECT_ROOT)

Project folder: E:\AssDS\Project


In [2]:
import re
import unicodedata

import pandas as pd
from datasets import load_dataset

pd.set_option("display.max_colwidth", 120)

## 1. Load and inspect the dataset

`load_dataset` downloads the data the first time. Later runs use the local cache.
The dataset contains one train split, so we will create our own train/test split
in the preprocessing notebook.

In [3]:
dataset = load_dataset("community-datasets/roman_urdu")
dataset

'[WinError 10013] An attempt was made to access a socket in a way forbidden by its access permissions' thrown while requesting HEAD https://huggingface.co/datasets/community-datasets/roman_urdu/resolve/a7bc39962a6bc489b7b6e4c11299e08ab149bb08/roman_urdu.py


Retrying in 1s [Retry 1/5].


'[WinError 10013] An attempt was made to access a socket in a way forbidden by its access permissions' thrown while requesting HEAD https://huggingface.co/datasets/community-datasets/roman_urdu/resolve/a7bc39962a6bc489b7b6e4c11299e08ab149bb08/roman_urdu.py


Retrying in 2s [Retry 2/5].


'[WinError 10013] An attempt was made to access a socket in a way forbidden by its access permissions' thrown while requesting HEAD https://huggingface.co/datasets/community-datasets/roman_urdu/resolve/a7bc39962a6bc489b7b6e4c11299e08ab149bb08/roman_urdu.py


Retrying in 4s [Retry 3/5].


... (4 outputs left out)

'[WinError 10013] An attempt was made to access a socket in a way forbidden by its access permissions' thrown while requesting HEAD https://huggingface.co/datasets/community-datasets/roman_urdu/resolve/a7bc39962a6bc489b7b6e4c11299e08ab149bb08/roman_urdu.py


Using the latest cached version of the dataset since community-datasets/roman_urdu couldn't be found on the Hugging Face Hub


Found the latest cached dataset configuration 'default' at E:\AssDS\Project\.cache\huggingface\datasets\community-datasets___roman_urdu\default\0.0.0\a7bc39962a6bc489b7b6e4c11299e08ab149bb08 (last modified on Mon Nov  9 16:23:33 2026).


DatasetDict({
    train: Dataset({
        features: ['sentence', 'sentiment'],
        num_rows: 20229
    })
})

In [4]:
train_data = dataset["train"]
print("Number of rows:", len(train_data))
print("Columns:", train_data.column_names)
print("Features:", train_data.features)
train_data[:5]

Number of rows: 20229
Columns: ['sentence', 'sentiment']
Features: {'sentence': Value('string'), 'sentiment': ClassLabel(names=['Positive', 'Negative', 'Neutral'])}


{'sentence': ['Sai kha ya her kisi kay bus ki bat nhi hai lakin main ki hal kal bi Aj aur aj bi sirf Aus say bus',
  'sahi bt h',
  'Kya bt hai,',
  'Wah je wah',
  'Are wha kaya bat hai'],
 'sentiment': [0, 0, 0, 0, 0]}

The sentiment is stored as a number internally. We use the dataset's own label
names to convert it into readable text instead of guessing what each number means.

In [5]:
df = train_data.to_pandas()
label_names = train_data.features["sentiment"].names
df["sentiment"] = df["sentiment"].map(
    {number: name for number, name in enumerate(label_names)}
)

df.head()

,sentence,sentiment
0,Sai kha ya her kisi kay bus ki bat nhi hai lakin main ki hal kal bi Aj aur aj bi sirf Aus say bus,Positive
1,sahi bt h,Positive
2,"Kya bt hai,",Positive
3,Wah je wah,Positive
4,Are wha kaya bat hai,Positive


In [6]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nClass counts:")
print(df["sentiment"].value_counts())

Shape: (20229, 2)

Data types:
sentence     object
sentiment    object
dtype: object

Missing values:
sentence     0
sentiment    0
dtype: int64

Class counts:
sentiment
Neutral     8929
Positive    6013
Negative    5287
Name: count, dtype: int64


## 2. Standardize text format

The function below makes formatting consistent without changing the meaning:

- normalize Unicode characters;
- remove invisible characters;
- strip spaces at both ends;
- replace repeated spaces/tabs/newlines with one space;
- create a lowercase version for duplicate checking and later modeling.

We keep the readable original sentence in `sentence`.

In [7]:
def clean_sentence(text):
    # Apply small, safe formatting fixes to one sentence.
    if pd.isna(text):
        return ""

    text = unicodedata.normalize("NFKC", str(text))
    text = re.sub(r"[\u200b-\u200d\ufeff]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df["sentence"] = df["sentence"].apply(clean_sentence)
df["clean_text"] = df["sentence"].str.lower()

df[["sentence", "clean_text", "sentiment"]].head()

,sentence,clean_text,sentiment
0,Sai kha ya her kisi kay bus ki bat nhi hai lakin main ki hal kal bi Aj aur aj bi sirf Aus say bus,sai kha ya her kisi kay bus ki bat nhi hai lakin main ki hal kal bi aj aur aj bi sirf aus say bus,Positive
1,sahi bt h,sahi bt h,Positive
2,"Kya bt hai,","kya bt hai,",Positive
3,Wah je wah,wah je wah,Positive
4,Are wha kaya bat hai,are wha kaya bat hai,Positive


## 3. Handle missing, empty, duplicate, and conflicting rows

Missing/empty text cannot teach a sentiment model anything, so it is removed.
Exact repeated text with the same label is also removed. If the same normalized
sentence has different labels, all copies are removed because there is no safe way
to decide which label is correct without manual relabeling.

In [8]:
starting_rows = len(df)

missing_label_count = df["sentiment"].isna().sum()
empty_text_count = df["clean_text"].eq("").sum()

df = df.dropna(subset=["sentiment"])
df = df[df["clean_text"] != ""].copy()

# A conflict means identical normalized text appears with more than one sentiment.
labels_per_text = df.groupby("clean_text")["sentiment"].nunique()
conflicting_texts = labels_per_text[labels_per_text > 1].index
conflicting_row_count = df["clean_text"].isin(conflicting_texts).sum()

df = df[~df["clean_text"].isin(conflicting_texts)].copy()

duplicate_count = df.duplicated(subset=["clean_text", "sentiment"]).sum()
df = df.drop_duplicates(subset=["clean_text", "sentiment"]).reset_index(drop=True)

print("Starting rows:", starting_rows)
print("Rows with missing sentiment:", missing_label_count)
print("Rows with empty text:", empty_text_count)
print("Rows removed because of conflicting labels:", conflicting_row_count)
print("Duplicate rows removed:", duplicate_count)
print("Final rows:", len(df))

Starting rows: 20229
Rows with missing sentiment: 0
Rows with empty text: 113
Rows removed because of conflicting labels: 106
Duplicate rows removed: 520
Final rows: 19490


## 4. Check text-length outliers

For numeric data, outliers are often removed. In text data, a very short or long
sentence can still be valid. We therefore inspect length outliers but keep them.
Only empty text has been removed.

In [9]:
df["character_count"] = df["clean_text"].str.len()
df["word_count"] = df["clean_text"].str.split().str.len()

q1 = df["word_count"].quantile(0.25)
q3 = df["word_count"].quantile(0.75)
iqr = q3 - q1
upper_limit = q3 + 1.5 * iqr
length_outliers = df[df["word_count"] > upper_limit]

print(df[["character_count", "word_count"]].describe().round(2))
print("\nIQR upper limit for word count:", upper_limit)
print("Long-text outliers kept:", len(length_outliers))

length_outliers.nlargest(5, "word_count")[["sentence", "sentiment", "word_count"]]

       character_count  word_count
count         19490.00    19490.00
mean             69.05       13.55
std              70.39       13.22
min               1.00        1.00
25%              27.00        6.00
50%              48.00       10.00
75%              89.00       17.00
max            2017.00      301.00

IQR upper limit for word count: 33.5
Long-text outliers kept: 1252


,sentence,sentiment,word_count
12544,Assalam o Alaikum! Main Ye 3 Cell Purchase kr chuka hun! Main Steps Main Review Krta Hun Is Ka! Unboxing krty he muj...,Positive,301
12033,"Assalam o alaikum! Dosto kal hum ny Federal or wo provinces jin mein abhi Ministerial posts ki upgradation nahi hoi,...",Negative,295
12083,kia bat hai ap ki. yar ap apna bata do k ap meret pai ay they. Khursheed shah ki comittee ny 90% darja chharm k log ...,Negative,210
12077,Dosto court ne up gradation case ka mohtasar faisala sunaya. UDC / LDC wala case jis banday nay kiya hoa ha us sy de...,Negative,200
15764,sb muslmanu ko.... dua ka sahy triqa sawab bakhshnay ka yai hai. k ya ilahi ju kuch b prha ish mai glty kotahy ko mu...,Positive,184


## 5. Final quality checks and save files

The raw CSV is useful for reproducibility. The cleaned CSV is the input to the EDA
and preprocessing notebooks. Length columns are exploratory features, not changes
to the original target.

In [10]:
valid_labels = {"Positive", "Negative", "Neutral"}

print("Missing values after cleaning:", int(df.isna().sum().sum()))
print("Remaining duplicate normalized texts:", int(df["clean_text"].duplicated().sum()))
print("Labels are valid:", set(df["sentiment"]).issubset(valid_labels))
print("Final shape:", df.shape)
print("\nFinal class counts:")
print(df["sentiment"].value_counts())

Missing values after cleaning: 0
Remaining duplicate normalized texts: 0
Labels are valid: True
Final shape: (19490, 5)

Final class counts:
sentiment
Neutral     8446
Positive    5810
Negative    5234
Name: count, dtype: int64


In [11]:
# Save the original converted data and the cleaned data.
raw_df = train_data.to_pandas()
raw_df["sentiment"] = raw_df["sentiment"].map(
    {number: name for number, name in enumerate(label_names)}
)

raw_path = DATA_RAW / "roman_urdu_raw.csv"
clean_path = DATA_PROCESSED / "roman_urdu_cleaned.csv"
summary_path = DATA_PROCESSED / "cleaning_summary.csv"

raw_df.to_csv(raw_path, index=False, encoding="utf-8")
df.to_csv(clean_path, index=False, encoding="utf-8")

cleaning_summary = pd.DataFrame(
    {
        "check": [
            "starting_rows",
            "missing_sentiment_rows",
            "empty_text_rows",
            "conflicting_label_rows",
            "duplicate_rows",
            "final_rows",
        ],
        "count": [
            starting_rows,
            missing_label_count,
            empty_text_count,
            conflicting_row_count,
            duplicate_count,
            len(df),
        ],
    }
)
cleaning_summary.to_csv(summary_path, index=False)

print("Saved:", raw_path)
print("Saved:", clean_path)
print("Saved:", summary_path)
cleaning_summary

Saved: E:\AssDS\Project\data\raw\roman_urdu_raw.csv
Saved: E:\AssDS\Project\data\processed\roman_urdu_cleaned.csv
Saved: E:\AssDS\Project\data\processed\cleaning_summary.csv


,check,count
0,starting_rows,20229
1,missing_sentiment_rows,0
2,empty_text_rows,113
3,conflicting_label_rows,106
4,duplicate_rows,520
5,final_rows,19490


## Cleaning conclusion

The resulting file has no missing labels, empty sentences, repeated normalized
sentences, or known label conflicts. Long messages remain because length alone
does not make a text record invalid. This avoids deleting useful language examples.